# Reproduce the familiarity/novelty manuscript figures and supplementary tables

This is the single reviewer-facing entry point. It validates the supplied Main Figure 1 design asset and regenerates Main Figures 2–5, Supplementary Figures 1–11, Supplementary Tables 1–16, and both unnumbered supplementary tabular displays.

Only anonymized preprocessed participant data and compact final analysis/display tables are used. Raw task files and task code are not required.

## Reproduction boundary

- Supplementary Figure 2 recomputes the N-F-1 participant-balanced trajectories and pointwise 95% t intervals from `data/n-f-1-participant-trajectories.csv`, then checks them against the frozen display summary.
- Supplementary Tables 4–7 recompute 66 correlation/BH-FDR rows from `data/n-f-2-to-5-participants.csv`; those values are cross-checked against the frozen Main Figure 3/4 display inputs before plotting.
- Table 8 refits its point estimates and validates the frozen 10,000-resample intervals.
- Trajectory/HLM, bootstrap, reliability, and sensitivity panels use compact plot-ready or result-level CSVs; upstream task preprocessing is intentionally outside this release.
- The plotting modules fix the active manuscript artboards, Arial typography, colors, markers, line styles, panel labels, and annotations.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'code'))

import reproduce_all
import validate

pd.set_option('display.max_columns', 40)
print('Repository root resolved; paths in released outputs remain relative.')

Repository root resolved; paths in released outputs remain relative.


## 1. Inspect and validate the anonymized participant cohorts

The released inputs retain the manuscript denominators: 15 N-F-1 pilot participants, 305 E2–E5 completers, 303 primary participants, and 153 primary E3/E5 matched participants.

In [2]:
experiment1 = pd.read_csv(ROOT / 'data' / 'n-f-1-participant-trajectories.csv')
participants = pd.read_csv(ROOT / 'data' / 'n-f-2-to-5-participants.csv')
participants['in_primary'] = participants['in_primary'].astype(str).str.lower().map({'true': True, 'false': False})
cohorts = (
    participants.groupby(['experiment', 'in_primary'], dropna=False)
    .size().rename('n').reset_index()
)
assert len(participants) == 305
assert int(participants['in_primary'].sum()) == 303
assert len(participants.query('experiment in [3, 5] and in_primary')) == 153
assert len(experiment1) == 1560
assert experiment1['experiment1_record_id'].nunique() == 15
assert set(experiment1['category']) == {'Face', 'Scenery', 'Geometry', 'Car'}
assert set(experiment1['comparison_position']) == set(range(1, 27))
display(experiment1.groupby('category', sort=False).agg(rows=('familiar_target_preference', 'size'), participants=('experiment1_record_id', 'nunique')))
display(cohorts)

,rows,participants
category,,
Face,390,15
Scenery,390,15
Geometry,390,15
Car,390,15


,experiment,in_primary,n
0,2,True,24
1,3,True,22
2,4,True,126
3,5,False,2
4,5,True,131


## 2. Trace every output to its compact inputs

The release manifest gives a direct, reviewer-readable route from each of the 34 figure/table components to its input CSV files or editable source and states whether the step recomputes an analysis, rerenders frozen results, or retains a canonical design asset.

In [3]:
output_inputs = pd.read_csv(ROOT / 'data' / 'output_input_manifest.csv')
data_dictionary = pd.read_csv(ROOT / 'data' / 'participant_data_dictionary.csv')
assert len(output_inputs) == 34
assert set(participants.columns) == set(data_dictionary['release_column'])
display(output_inputs)
display(data_dictionary)

,output_file,input_files,reproduction_level,note
0,outputs/figures/main/main_figure_01.pdf,source/figure_01/main_figure_01_editable.pptx,editable design source + canonical PDF,Conceptual main-manuscript figure; no statisti...
1,outputs/figures/main/main_figure_02.pdf,data/main/figure02/observed_trajectory.csv; da...,plot-data rendering,Active main-manuscript design and artboard
2,outputs/figures/main/main_figure_03.pdf,data/n-f-2-to-5-participants.csv; data/main/fi...,recomputed correlation family + frozen display...,Active main-manuscript design and artboard
3,outputs/figures/main/main_figure_04.pdf,data/n-f-2-to-5-participants.csv; data/main/fi...,recomputed correlation family + frozen display...,Active main-manuscript design and artboard
4,outputs/figures/main/main_figure_05.pdf,data/main/figure05/moderator_screen_bh12.csv; ...,frozen model/result rendering,Active main-manuscript design and artboard
5,outputs/figures/supplement/supplementary_figur...,data/supplement/figure_01_demographics/age_poi...,aggregate/plot-data rendering,Active Supplementary Information design and ar...
6,outputs/figures/supplement/supplementary_figur...,data/n-f-1-participant-trajectories.csv; data/...,recomputed participant-balanced trajectory + f...,Active Supplementary Information design and ar...
7,outputs/figures/supplement/supplementary_figur...,data/supplement/figure_03_experiment4_trajecto...,aggregate/plot-data rendering,Active Supplementary Information design and ar...
8,outputs/figures/supplement/supplementary_figur...,data/supplement/figure_04_fni_patterns/margina...,aggregate/plot-data rendering,Active Supplementary Information design and ar...
9,outputs/figures/supplement/supplementary_figur...,data/supplement/figure_05_selected_profiles/se...,aggregate/plot-data rendering,Active Supplementary Information design and ar...


,release_column,original_or_manuscript_name,data_type,scale_or_theoretical_range,observed_range_or_levels,nonmissing_n,description
0,analysis_record_id,release pseudonym,string,A001--A305,305 unique release values,305,Join key for released analysis tables; not a p...
1,experiment,experiment,integer,2--5,2 to 5,305,Experiment number.
2,mode,setting,category,lab or online,lab; online,305,Data-collection mode.
3,in_primary,QC-clean membership,boolean,True or False,False; True,305,True for the frozen primary integrated cohort ...
4,face_fni,Face FNI,number,theoretical -3 to +3,-1.86 to 2.63853,305,Participant Face familiarity/novelty index; hi...
5,geometry_fni,Geometry FNI,number,theoretical -3 to +3,-1.9025 to 2.42059,305,Participant Geometry familiarity/novelty index...
6,scenery_fni,Scenery FNI,number,theoretical -3 to +3,-2.27941 to 1.94118,305,Participant Scenery familiarity/novelty index;...
7,aq_score,AQ_score,integer,0--50,5 to 43,305,Autism-Spectrum Quotient total; higher values ...
8,daily_familiarity,4_NF-daily / 4_nf_daily,number,-3 to +3,-3 to 3,305,Mean of eight daily novelty/familiarity items;...
9,maemuki_score,Maemuki_score,number,approximately -3 to +3,-1.0844 to 2.416,305,Composite positive/forward psychological orien...


## 3. Regenerate every figure and table

The command below uses the same functions as `python code/reproduce_all.py`. Existing outputs are overwritten.

In [4]:
import json
import subprocess

completed = subprocess.run(
    [sys.executable, str(ROOT / 'code' / 'reproduce_all.py')],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)
run = json.loads((ROOT / 'outputs' / 'run_manifest.json').read_text())

{
  "main_figures": [
    "outputs/figures/main/main_figure_01.pdf",
    "outputs/figures/main/main_figure_02.pdf",
    "outputs/figures/main/main_figure_03.pdf",
    "outputs/figures/main/main_figure_04.pdf",
    "outputs/figures/main/main_figure_05.pdf"
  ],
  "supplementary_figures": [
    "outputs/figures/supplement/supplementary_figure_01.pdf",
    "outputs/figures/supplement/supplementary_figure_02.pdf",
    "outputs/figures/supplement/supplementary_figure_03.pdf",
    "outputs/figures/supplement/supplementary_figure_04.pdf",
    "outputs/figures/supplement/supplementary_figure_05.pdf",
    "outputs/figures/supplement/supplementary_figure_06.pdf",
    "outputs/figures/supplement/supplementary_figure_07.pdf",
    "outputs/figures/supplement/supplementary_figure_08.pdf",
    "outputs/figures/supplement/supplementary_figure_09.pdf",
    "outputs/figures/supplement/supplementary_figure_10.pdf",
    "outputs/figures/supplement/supplementary_figure_11.pdf"
  ],
  "tables": [
    "outpu

## 4. Supplementary results: Tables 1–16 and inline displays

The tables below are the supplementary results underlying the publication-formatted TeX outputs. Tables 4–8 use the generated result CSVs created by the reproduction pipeline; the remaining tables use the released aggregate, metadata, or frozen-result inputs. Together they show all 16 numbered Supplementary Tables and the two unnumbered supplementary tabular displays directly in the notebook.

In [5]:
from IPython.display import Markdown, display

supplementary_results = [
    ('Supplementary Table 1. Analysis populations and manuscript roles', [
        ('', 'data/tables/table01_analysis_populations.csv'),
    ]),
    ('Supplementary Table 2. Experiment-specific visual-task implementations', [
        ('', 'data/tables/table02_visual_task_implementations.csv'),
    ]),
    ('Supplementary Table 3. Participant demographics in the primary E2–E5 analysis set', [
        ('Age and recorded gender at birth', 'data/tables/table03_age_gender.csv'),
        ('Recorded race response', 'data/tables/table03_race.csv'),
    ]),
    ('Supplementary Table 4. Matched visual-category FNI by questionnaire associations', [
        ('', 'outputs/tables/supplementary_table_04_data.csv'),
    ]),
    ('Supplementary Table 5. Matched visual-category FNI by Daily familiarity item associations', [
        ('', 'outputs/tables/supplementary_table_05_data.csv'),
    ]),
    ('Supplementary Table 6. Pooled questionnaire-pair associations', [
        ('', 'outputs/tables/supplementary_table_06_data.csv'),
    ]),
    ('Supplementary Table 7. Pooled Daily familiarity item by questionnaire associations', [
        ('', 'outputs/tables/supplementary_table_07_data.csv'),
    ]),
    ('Supplementary Table 8. Questionnaire relationships in the joint model', [
        ('', 'outputs/tables/supplementary_table_08_data.csv'),
    ]),
    ('Supplementary Table 9. Primary category trajectories and pairwise differences', [
        ('', 'data/tables/table09_primary_trajectories.csv'),
    ]),
    ('Supplementary Table 10. Sensitivity analyses for the category trajectories', [
        ('', 'data/tables/table10_trajectory_sensitivities.csv'),
    ]),
    ('Supplementary Table 11. Overall-moderator trajectory family and item-aware sensitivity', [
        ('', 'data/tables/table11_trait_moderation.csv'),
    ]),
    ('Supplementary Table 12. Joint-trait trajectory-moderation sensitivity', [
        ('', 'data/tables/table12_joint_trait_moderation.csv'),
    ]),
    ('Supplementary Table 13. Daily-item moderation family and item-aware sensitivity', [
        ('', 'data/tables/table13_daily_item_moderation.csv'),
    ]),
    ('Supplementary Table 14. Cross-subcategory stability', [
        ('', 'data/tables/table14_cross_subcategory_stability.csv'),
    ]),
    ('Supplementary Table 15. Task-pool stimulus taxonomy across Experiments 1–5', [
        ('', 'data/tables/table15_stimulus_taxonomy.csv'),
    ]),
    ('Supplementary Table 16. Questionnaire-derived features used in Experiments 2–5', [
        ('', 'data/tables/table16_questionnaire_features.csv'),
    ]),
    ('Supplementary inline result 1. Main-text to Supplementary Information cross-reference', [
        ('', 'data/tables/inline01_cross_reference.csv'),
    ]),
    ('Supplementary inline result 2. Face-FNI adjusted and joint-model robustness', [
        ('', 'data/tables/inline02_face_fni_adjusted_joint.csv'),
    ]),
]

assert len(supplementary_results) == 18
for title, sections in supplementary_results:
    display(Markdown(f'### {title}'))
    for section_label, relative_path in sections:
        result_path = ROOT / relative_path
        assert result_path.is_file(), result_path
        if section_label:
            display(Markdown(f'**{section_label}**'))
        display(pd.read_csv(result_path))

print('Displayed all 16 numbered Supplementary Tables and both supplementary inline results.')

### Supplementary Table 1. Analysis populations and manuscript roles

,experiment,setting,completers,qc_clean,manuscript_role
0,1,Laboratory,15,--,Pilot; excluded from current analyses
1,2,Laboratory,24,24,Integrated questionnaire/sample scope
2,3,Laboratory,22,22,Integrated scope; matched visual analysis
3,4,Online,126,126,Integrated scope; descriptive target-shift ana...
4,5,Online,133,131,Integrated scope; matched visual analysis
5,3 and 5 combined,Mixed,155,153,Matched trial-level cohort
6,2--5 combined,Mixed,305,303,Primary integrated questionnaire/sample scope


### Supplementary Table 2. Experiment-specific visual-task implementations

,study,setting_blocks,rating_sequence,preference_sequence
0,1,"Laboratory; two sessions across Face, Scenery,...",Nominally 27 ratings; the archived Diagonal-SU...,26 previously encountered comparisons per subc...
1,2,"Laboratory; eight blocks across Face, Scenery,...",19 rating records per block in all 24 raw expo...,18 comparisons per block; integrated questionn...
2,3,"Laboratory; 12 blocks across Face, Scenery and...",19 ratings; nominal median target (rank 10 of 19),18 comparisons per block; matched visual analysis
3,4,"Online; 12 blocks across Face, Scenery and Geo...",19 ratings; implementation error selected the ...,18 comparisons per block; descriptive design-d...
4,5,"Online; 12 blocks across Face, Scenery and Geo...",19 ratings; corrected median target (rank 10 o...,18 comparisons per block; matched visual analysis


### Supplementary Table 3. Participant demographics in the primary E2–E5 analysis set

...

### Supplementary inline result 1. Main-text to Supplementary Information cross-reference

,main_text_result,supplement_location,interpretation_boundary
0,Study design (\textcolor{blue}{Fig.~1}) and Me...,"Cohorts and demographics (Section~2), addition...","Core outcome, FNI and trajectory definitions a..."
1,Visual trajectories (\textcolor{blue}{Fig.~5a}),"Model estimates, planned contrasts and sensiti...",Primary inference uses the matched Experiment~...
2,Individual profiles (\textcolor{blue}{Fig.~2b}),Category-pattern counts and selected-case comp...,Descriptive only; no discrete participant-type...
3,Matched FNI associations (\textcolor{blue}{Fig...,"Reliability, complete association families and...",The FNI is defined in the main Methods; these ...
4,Integrated questionnaire associations (\textco...,"Complete pooled correlations, the joint questi...",\textbf{Secondary} or exploratory and cross-se...
5,Trajectory moderators (\textcolor{blue}{Fig.~5...,"Complete moderator families, item-aware infere...",\textit{Exploratory}; Daily-item candidates ar...
6,Questionnaire battery (\textcolor{blue}{Fig.~1...,Reconstruction and reliability (Section~8.2; S...,Measurement implementations differed across ex...


### Supplementary inline result 2. Face-FNI adjusted and joint-model robustness

,measure,adjusted_r,q_bh,joint_beta,p_value
0,AQ,-0.250,0.0132,-0.143,0.158
1,Maemuki,0.263,0.0132,0.115,0.384
2,Flow,0.208,0.0388,0.087,0.412


Displayed all 16 numbered Supplementary Tables and both supplementary inline results.


## 5. Verify analysis consistency, figure content, tables, and privacy minimization

Acceptance requires the exact artboards, expected extracted text, the declared raster policy, all table fragments, agreement between recomputed and frozen correlation families, and the namespace/minimization scan. Analysis figures remain vector-only; Main Figure 1 permits its embedded stimulus images. Binary PDF hashes remain provenance only because harmless metadata and object-order differences change file bytes.

In [6]:
figure_validation = pd.read_csv(ROOT / 'outputs' / 'figure_validation.csv')
table_validation = pd.read_csv(ROOT / 'outputs' / 'table_validation.csv')
privacy_validation = pd.read_csv(ROOT / 'outputs' / 'data_privacy_validation.csv')
analysis_crosscheck = pd.read_csv(ROOT / 'outputs' / 'analysis_crosscheck.csv')

assert figure_validation['artboard_ok'].all()
assert figure_validation['content_ok'].all()
assert figure_validation['raster_policy_ok'].all()
assert table_validation['exists'].all()
assert privacy_validation['public_identifier_check'].all()
assert privacy_validation['namespace_and_minimization_check'].all()
assert analysis_crosscheck['within_tolerance'].all()

display(figure_validation[['figure', 'observed_width_pt', 'observed_height_pt', 'artboard_ok', 'required_text_found', 'vector_only', 'raster_policy_ok', 'content_ok']])
display(analysis_crosscheck)
print(f"Validated {len(table_validation)} table fragments and {len(privacy_validation)} public CSV files.")

,figure,observed_width_pt,observed_height_pt,artboard_ok,required_text_found,vector_only,raster_policy_ok,content_ok
0,Main Figure 1,518.000,346.000000,True,True,False,True,True
1,Main Figure 2,345.600,324.480000,True,True,True,True,True
2,Main Figure 3,345.600,320.400000,True,True,True,True,True
3,Main Figure 4,345.600,450.000000,True,True,True,True,True
4,Main Figure 5,345.600,357.165354,True,True,True,True,True
5,Supplementary Figure 1,510.236,459.213000,True,True,True,True,True
6,Supplementary Figure 2,509.760,297.600000,True,True,True,True,True
7,Supplementary Figure 3,510.236,297.638000,True,True,True,True,True
8,Supplementary Figure 4,510.236,566.929000,True,True,True,True,True
9,Supplementary Figure 5,518.400,146.880000,True,True,True,True,True


,analysis_family,matched_rows,expected_rows,missing_keys,all_n_match,max_rho_absolute_error,max_q_absolute_error,tolerance,within_tolerance
0,Main Figure 3 BH-12 / Supplementary Table 4,12,12,NaN,True,4.132387e-07,3.383250e-07,0.0005,True
1,Main Figure 3 BH-24 / Supplementary Table 5,24,24,NaN,True,4.947821e-05,4.753539e-06,0.0005,True
2,Main Figure 4 BH-6 / Supplementary Table 6,6,6,NaN,True,0.000000e+00,0.000000e+00,0.0005,True
3,Main Figure 4 BH-24 / Supplementary Table 7,24,24,NaN,True,0.000000e+00,0.000000e+00,0.0005,True


Validated 18 table fragments and 56 public CSV files.


## Interpretation and release notes

The N-F-1 IDs `E1A001`–`E1A015` are release-only pseudonyms and are not linked to archived filenames or later experiments. The manuscript profile IDs `P092` and `P101` are author-selected descriptive examples, not clusters or participant types. They map to release analysis records `A242` and `A251`; the two namespaces must not be joined by their numeric suffix. Figure 5 keeps the matched E3/E5 trajectory cohort separate from the pooled questionnaire cohort, and all multiplicity families and sensitivity labels are retained.

The automated privacy checks support data minimization but do not establish legal or ethical de-identification. Obtain corresponding-author/data-controller approval under the applicable consent, ethics/IRB, and institutional data-sharing terms before publishing participant-level derived rows or age points.